In [1]:
import PyPDF2
import re
from pathlib import Path
import sys
import fitz          # PyMuPDF -- image extraction
import pdfplumber     # table extraction
import json
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config.db import get_connection

conn2 = get_connection()

conn2.autocommit = True

In [3]:
# from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

pdf_path = PROJECT_ROOT / "docs" / "2023-annual-report-truncated.pdf"
reader = PyPDF2.PdfReader(str(pdf_path))

In [4]:
def generate_document_id(conn2):
    with conn2.cursor() as cur:
        cur.execute("""
            SELECT id
            FROM "Document"
            ORDER BY id DESC
            LIMIT 1
        """)

        row = cur.fetchone()

    if row is None:
        return "DOC000001"

    last_number = int(re.search(r"\d+", row[0]).group())

    return f"DOC{last_number + 1:06d}"

In [5]:
document_id = generate_document_id(conn2)

document_title = pdf_path.stem
source_file = pdf_path.name
total_pages = len(reader.pages)

with conn2.cursor() as cur:

    cur.execute(
        """
        INSERT INTO "Document"
        (
            id,
            title,
            "sourceFile",
            "totalPages"
        )
        VALUES
        (%s,%s,%s,%s)
        """,
        (
            document_id,
            document_title,
            source_file,
            total_pages,
        ),
    )

conn2.commit()

print("Document Stored Successfully")
print(document_id)

Document Stored Successfully
DOC000001


In [6]:
with conn2.cursor() as cur:

    for page_number, page in enumerate(reader.pages, start=1):

        page_id = f"{document_id}_P{page_number:03d}"

        text = page.extract_text()

        if text is None:
            text = ""

        text = text.strip()

        token_count = len(text.split())

        cur.execute(
            """
            INSERT INTO "Page"
            (
                id,
                "documentId",
                "pageNumber",
                content,
                "tokenCount"
            )
            VALUES
            (%s,%s,%s,%s,%s)
            """,
            (
                page_id,
                document_id,
                page_number,
                text,
                token_count,
            ),
        )

conn2.commit()

print(f"{len(reader.pages)} Pages Stored Successfully")

50 Pages Stored Successfully


In [7]:
with conn2.cursor() as cur:

    cur.execute("""
        SELECT
            id,
            "pageNumber",
            "tokenCount"
        FROM "Page"
        WHERE "documentId"=%s
        ORDER BY "pageNumber"
    """, (document_id,))

    rows = cur.fetchall()

for row in rows[:5]:
    print(row)

('DOC000001_P001', 1, 30)
('DOC000001_P002', 2, 0)
('DOC000001_P003', 3, 260)
('DOC000001_P004', 4, 88)
('DOC000001_P005', 5, 118)


In [8]:
IMAGE_DIR = PROJECT_ROOT / "docs" / "extracted_images" / document_id
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

CAPTION_RE = re.compile(r'^(Figure|Table)\s+(\d+\.\d+)\.\s*(.*)$')
NOTE_RE = re.compile(r'^(Note|Source)\s*:', re.IGNORECASE)

In [9]:
def load_page_id_map(conn2, document_id: str) -> dict:
    with conn2.cursor() as cur:
        cur.execute('SELECT "pageNumber", id FROM "Page" WHERE "documentId" = %s', (document_id,))
        rows = cur.fetchall()
    return {page_number: page_id for page_number, page_id in rows}
 
 
page_id_map = load_page_id_map(conn2, document_id)

In [10]:
# Find every caption line ("Figure N.N. ..." / "Table N.N. ...") on a page,
# with its exact bbox, using PyMuPDF's own text layout.
# ---------------------------------------------------------------------------
def find_captions(page) -> list:
    captions = []
    page_dict = page.get_text("dict")
    for block in page_dict.get("blocks", []):
        for line in block.get("lines", []):
            text = "".join(span["text"] for span in line["spans"]).strip()
            if not text:
                continue
            m = CAPTION_RE.match(text)
            if m:
                x0 = min(span["bbox"][0] for span in line["spans"])
                y0 = min(span["bbox"][1] for span in line["spans"])
                x1 = max(span["bbox"][2] for span in line["spans"])
                y1 = max(span["bbox"][3] for span in line["spans"])
                captions.append({
                    "kind":   m.group(1),          # "Figure" or "Table"
                    "number": m.group(2),          # e.g. "1.1"
                    "title":  m.group(3).strip(),
                    "rect":   fitz.Rect(x0, y0, x1, y1),
                })
    return captions

def find_note_lines(page) -> list:
    """Bboxes of 'Note:' / 'Source:' lines -- these mark the bottom of a figure."""
    notes = []
    page_dict = page.get_text("dict")
    for block in page_dict.get("blocks", []):
        for line in block.get("lines", []):
            text = "".join(span["text"] for span in line["spans"]).strip()
            if NOTE_RE.match(text):
                x0 = min(span["bbox"][0] for span in line["spans"])
                y0 = min(span["bbox"][1] for span in line["spans"])
                x1 = max(span["bbox"][2] for span in line["spans"])
                notes.append(fitz.Rect(x0, y0, x1, y0 + 1))
    return notes

In [11]:
# Figures -- crop from caption down to the matching Note/Source line
# ---------------------------------------------------------------------------
def _content_rects(page) -> list:
    """All raster-image and vector-drawing rects on the page, for widening
    the crop to the real chart extent."""
    rects = []
    for img in page.get_images(full=True):
        for r in page.get_image_rects(img[0]):
            rects.append(r)
    for d in page.get_drawings():
        if d.get("rect") and not d["rect"].is_empty:
            rects.append(d["rect"])
    return rects

In [12]:
def build_figure_crop(page, caption: dict, next_caption_top: float) -> fitz.Rect:
    page_w = page.rect.width
 
    # bottom bound: nearest Note/Source line below the caption, else the next
    # caption on the page, else a generous fallback
    notes_below = [n.y0 for n in find_note_lines(page) if n.y0 > caption["rect"].y1]
    if notes_below:
        bottom = min(notes_below) + 40   # include the note text itself
    elif next_caption_top is not None:
        bottom = next_caption_top - 5
    else:
        bottom = min(caption["rect"].y1 + 320, page.rect.height - 20)
 
    top = caption["rect"].y0 - 4
 
    # horizontal bound: single-column figures (like Figure 1.1) span close to
    # full page width; two-column figures (like Figure 2.1) are indented and
    # narrower -- use the caption's own width as a signal, then widen to
    # whatever chart content actually falls in this vertical band
    if caption["rect"].width > page_w * 0.55:
        left, right = 30, page_w - 30
    else:
        left = caption["rect"].x0 - 10
        right = caption["rect"].x0 + (page_w * 0.46)
 
    band = fitz.Rect(0, top, page_w, bottom)
    for r in _content_rects(page):
        if r.intersects(band) and r.width > 20 and r.height > 20:
            left = min(left, r.x0 - 5)
            right = max(right, r.x1 + 5)
 
    left = max(left, 0)
    right = min(right, page_w)
    return fitz.Rect(left, top, right, bottom)

In [13]:
def extract_figures(pdf_path: Path, document_id: str, page_id_map: dict) -> list:
    doc = fitz.open(str(pdf_path))
    figures = []
 
    for page_number in range(1, doc.page_count + 1):
        page = doc[page_number - 1]
        captions = [c for c in find_captions(page) if c["kind"] == "Figure"]
        captions.sort(key=lambda c: c["rect"].y0)
 
        for i, cap in enumerate(captions):
            next_top = captions[i + 1]["rect"].y0 if i + 1 < len(captions) else None
            crop = build_figure_crop(page, cap, next_top)
 
            pix = page.get_pixmap(matrix=fitz.Matrix(2.5, 2.5), clip=crop)
            filename = f"{document_id}_p{page_number:03d}_fig{cap['number'].replace('.', '_')}.png"
            out_path = IMAGE_DIR / filename
            pix.save(str(out_path))
 
            figures.append({
                "id":           f"{document_id}_F{cap['number'].replace('.', '_')}",
                "documentId":   document_id,
                "pageId":       page_id_map.get(page_number),
                "pageNumber":   page_number,
                "figureNumber": f"Figure {cap['number']}",
                "title":        cap["title"],
                "imagePath":    str(out_path),
                "bbox":         {"x0": crop.x0, "y0": crop.y0, "x1": crop.x1, "y1": crop.y1},
            })
 
    doc.close()
    return figures

In [14]:
# Tables -- keep only pdfplumber tables with a matching "Table N.N." caption
# ---------------------------------------------------------------------------
def table_to_markdown(rows: list) -> str:
    rows = [[c if c is not None else "" for c in row] for row in rows]
    header, *body = rows
    lines = ["| " + " | ".join(str(c) for c in header) + " |"]
    lines.append("|" + "|".join(["---"] * len(header)) + "|")
    for row in body:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    return "\n".join(lines)
 
 
def extract_tables(pdf_path: Path, document_id: str, page_id_map: dict) -> list:
    doc = fitz.open(str(pdf_path))
    tables = []
 
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_number, plumber_page in enumerate(pdf.pages, start=1):
            fitz_page = doc[page_number - 1]
            captions = [c for c in find_captions(fitz_page) if c["kind"] == "Table"]
            if not captions:
                continue  # no table caption on this page -> nothing to keep
 
            found_tables = []
            for strategy in (
                {"vertical_strategy": "lines", "horizontal_strategy": "lines"},
                {"vertical_strategy": "text", "horizontal_strategy": "text",
                 "snap_tolerance": 5, "intersection_tolerance": 5},
            ):
                try:
                    for t in plumber_page.find_tables(table_settings=strategy):
                        rows = t.extract()
                        if rows and len(rows) >= 2:
                            found_tables.append((t.bbox, rows))
                except Exception:
                    continue
 
            for cap in captions:
                cap_bottom = cap["rect"].y1
                # match the table whose top is closest below this caption
                candidates = [
                    (bbox, rows) for bbox, rows in found_tables
                    if bbox[1] >= cap_bottom - 5 and bbox[1] - cap_bottom < 60
                ]
                if not candidates:
                    continue
                bbox, rows = min(candidates, key=lambda c: c[0][1])
 
                tables.append({
                    "id":          f"{document_id}_T{cap['number'].replace('.', '_')}",
                    "documentId":  document_id,
                    "pageId":      page_id_map.get(page_number),
                    "pageNumber":  page_number,
                    "tableNumber": f"Table {cap['number']}",
                    "title":       cap["title"],
                    "markdown":    table_to_markdown(rows),
                })
 
    doc.close()
    return tables

In [15]:
# Persist
# ---------------------------------------------------------------------------
def store_figures(conn2, figures: list):
    with conn2.cursor() as cur:
        for fig in figures:
            cur.execute(
                '''INSERT INTO "Figure" (id, "documentId", "pageId", "figureNumber", title, "imagePath", bbox)
                   VALUES (%s,%s,%s,%s,%s,%s,%s)''',
                (fig["id"], fig["documentId"], fig["pageId"], fig["figureNumber"], fig["title"],
                 fig["imagePath"], json.dumps(fig["bbox"]) if fig["bbox"] else None),
            )
    conn2.commit()
 
 
def store_tables(conn2, tables: list):
    with conn2.cursor() as cur:
        for tbl in tables:
            cur.execute(
                '''INSERT INTO "Table" (id, "documentId", "pageId", "tableNumber", title, markdown)
                   VALUES (%s,%s,%s,%s,%s,%s)''',
                (tbl["id"], tbl["documentId"], tbl["pageId"], tbl["tableNumber"], tbl["title"], tbl["markdown"]),
            )
    conn2.commit()

In [16]:
# Run
# ---------------------------------------------------------------------------
with conn2.cursor() as cur:
    cur.execute('DELETE FROM "Figure" WHERE "documentId" = %s', (document_id,))
    cur.execute('DELETE FROM "Table" WHERE "documentId" = %s', (document_id,))
conn2.commit()
 
figures = extract_figures(pdf_path, document_id, page_id_map)
tables = extract_tables(pdf_path, document_id, page_id_map)
 
store_figures(conn2, figures)
store_tables(conn2, tables)
 
print(f"{len(figures)} figures stored (caption-anchored)")
for f in figures[:5]:
    print(f"   {f['figureNumber']}: {f['title'][:60]}")
print(f"{len(tables)} tables stored (caption-anchored)")
for t in tables[:5]:
    print(f"   {t['tableNumber']}: {t['title'][:60]}")

12 figures stored (caption-anchored)
   Figure 1.1: The Federal Reserve System’s unique structure ensures broad 
   Figure 2.1: Personal consumption expenditures
   Figure 2.2: Nonfarm payroll employment
   Figure 2.3: Unemployment rate, by race and ethnicity
   Figure 2.4: Selected interest rates
0 tables stored (caption-anchored)
